In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Binarizer, KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

warnings.filterwarnings("ignore")

# Preparation

1. Load train.csv and test.csv
              ↓
2. Inspect data
   - missing values
   - duplicates
   - distributions
   - skewness
              ↓
3. Train/validation split
              ↓
4. Data cleaning + preprocessing
   - median imputation
   - most-frequent imputation
   - categorical encoding
   - scaling
              ↓
5. Feature engineering
   - FareLog
   - FamilySize
   - IsAlone
   - Title
   - Deck
              ↓
6. Train models
   - Naive Bayes
   - Linear Regression
   - Ridge
   - LASSO
              ↓
7. Evaluate on validation set
              ↓
8. Choose best model
              ↓
9. Retrain best model on ALL train.csv
              ↓
10. Predict test.csv


We calculate preprocessing rules (such as the median for imputation) only from the training data and apply those same rules to the validation data because the validation set should simulate completely new, unseen data; if we calculate its own median, we are allowing the validation data to influence the preprocessing and making our evaluation less fair. 
Imagine tomorrow you receive one new passenger:
Age = NaN

Where would you get the median?
You don't have a new validation dataset containing hundreds of passengers from which you can calculate a new median.
You only have the rules learned from your training data.

So your deployed system has to say:
"If Age is missing, use the median I learned during training."
That is why validation should simulate that situation.

In [ ]:
# Configuration

RANDOM_STATE = 42
TEST_SIZE = 0.20

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)

# Load the data and quick explore

later: check if the character columns have no numbers and number columns have no letters!!!!!! Clean if possible

In [ ]:
# Load the data

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")

print("TRAIN INFO")
display(train.describe(include="all"))

print("TEST INFO")
display(test.describe(include="all"))

In [ ]:
print("Missing values in train:")
display(train.isnull().sum())

print("\nMissing values in test:")
display(test.isnull().sum())

print("\nDuplicate rows:")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())

# Quick Data cleaning
Cleaning is performed before the train/validation split for operations that do not learn parameters from the data. Missing-value imputation, scaling, and encoding are handled later inside model-specific pipelines so that those parameters are learned from the training split only.

In [ ]:
# Standardize text formatting
for df in [train, test]:

    for col in ["Name", "Sex", "Ticket", "Cabin", "Embarked"]:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    df["Sex"] = df["Sex"].str.lower()
    df["Embarked"] = df["Embarked"].str.upper()

# Replace impossible numerical values with missing
for df in [train, test]:

    df.loc[df["Age"] < 0, "Age"] = np.nan
    df.loc[df["Fare"] < 0, "Fare"] = np.nan
    df.loc[df["SibSp"] < 0, "SibSp"] = np.nan
    df.loc[df["Parch"] < 0, "Parch"] = np.nan

# TRAIN / VALIDATION SPLIT

In [ ]:
# Separate target from predictors
X = train.drop(columns=["Survived"])
y = train["Survived"]

# Split the labeled training data
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)

# Training Set Data Exploration -- More Detailed

In [ ]:
# ============================================
# DATA EXPLORATION — TRAINING SET
# ============================================

print("TRAINING SET INFO")
print("=" * 60)
X_train.info()

In [ ]:
# ============================================
# MISSING VALUES — TRAINING SET
# ============================================

missing_train = pd.DataFrame({
    "Missing Count": X_train.isnull().sum(),
    "Missing Percentage": X_train.isnull().mean() * 100
})

display(
    missing_train
    .sort_values("Missing Percentage", ascending=False)
)

In [ ]:
# ============================================
# CATEGORICAL VARIABLES — TRAINING SET
# ============================================

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

for ax, column in zip(axes, categorical_features):

    # Counts
    counts = X_train[column].value_counts(dropna=False)

    # Percentages
    percentages = (
        X_train[column]
        .value_counts(dropna=False, normalize=True)
        * 100
    )

    # Convert missing values to a readable label
    labels = [
        "Missing" if pd.isna(value) else str(value)
        for value in counts.index
    ]

    # Plot
    bars = ax.bar(
        labels,
        counts.values
    )

    ax.set_title(f"{column} Distribution")
    ax.set_xlabel(column)
    ax.set_ylabel("Number of Passengers")

    # Add count and percentage to each bar
    for bar, count, percentage in zip(
        bars,
        counts.values,
        percentages.values
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            f"{count} ({percentage:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=10
        )

    # Leave room for labels
    ax.set_ylim(
        0,
        max(counts.values) * 1.18
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# SKEWNESS — TRAINING SET
# ============================================

numeric_features_to_check = [
    "Age",
    "Fare",
    "SibSp",
    "Parch"
]

# --------------------------------------------
# Calculate skewness
# --------------------------------------------

skewness_results = pd.DataFrame({
    "Feature": numeric_features_to_check,
    "Skewness": [
        X_train[column].skew()
        for column in numeric_features_to_check
    ]
})

skewness_results["Absolute Skewness"] = (
    skewness_results["Skewness"].abs()
)

skewness_results["Interpretation"] = np.where(
    skewness_results["Absolute Skewness"] < 0.5,
    "Approximately symmetric",
    np.where(
        skewness_results["Absolute Skewness"] < 1,
        "Moderately skewed",
        "Strongly skewed"
    )
)

# Display results
display(
    skewness_results.sort_values(
        "Absolute Skewness",
        ascending=False
    )
)

# --------------------------------------------
# Visualize distributions
# --------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8)
)

axes = axes.flatten()

for i, column in enumerate(numeric_features_to_check):

    axes[i].hist(
        X_train[column].dropna(),
        bins=30
    )

    skew_value = X_train[column].skew()

    axes[i].set_title(
        f"{column} Distribution\n"
        f"Skewness = {skew_value:.3f}"
    )

    axes[i].set_xlabel(column)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# TARGET DISTRIBUTION — TRAINING SET
# ============================================

survived_counts = y_train.value_counts().sort_index()
survived_percentages = (
    y_train.value_counts(normalize=True)
    .sort_index()
    * 100
)

labels = ["Did Not Survive", "Survived"]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    labels,
    survived_counts.values
)

plt.title("Survival Distribution — Training Set")
plt.xlabel("Survival Status")
plt.ylabel("Number of Passengers")

for bar, count, percentage in zip(
    bars,
    survived_counts.values,
    survived_percentages.values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"{count} ({percentage:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.ylim(0, max(survived_counts.values) * 1.15)
plt.tight_layout()
plt.show()

# Feature Engineering 

In [ ]:
# FEATURE ENGINEERING

def feature_engineer(df):
    df = df.copy()

    # ----------------------------
    # Fare transformation
    # ----------------------------
    df["FareLog"] = np.log1p(df["Fare"])

    # ----------------------------
    # Family features
    # ----------------------------
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    df["IsAlone"] = (
        df["FamilySize"] == 1
    ).astype(int)

    # ----------------------------
    # Title from Name
    # ----------------------------
    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
    )

    # Standardize titles
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    # Keep common titles and group the rest
    df["Title"] = df["Title"].where(
        df["Title"].isin([
            "Mr",
            "Miss",
            "Mrs",
            "Master"
        ]),
        "Rare"
    )

    # ----------------------------
    # Deck from Cabin
    # ----------------------------
    df["Deck"] = (
        df["Cabin"]
        .fillna("Unknown")
        .str[0]
    )

    # ----------------------------
    # Cabin known indicator
    # ----------------------------
    df["CabinKnown"] = (
        df["Cabin"].notna()
    ).astype(int)

    # ----------------------------
    # Ticket prefix
    # ----------------------------
    df["TicketPrefix"] = (
        df["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[\s./]+", "", regex=True)
        .str.upper()
    )

    df["TicketPrefix"] = df["TicketPrefix"].replace(
        "",
        "NUMERIC"
    )

    return df


X_train_fe = feature_engineer(X_train)
X_val_fe = feature_engineer(X_val)   # Feature engineering that uses fixed rules -> okay on val.

print("Training data after feature engineering:")
print(X_train_fe.shape)

print("\nValidation data after feature engineering:")
print(X_val_fe.shape)

# Preprocessing and more cleaning for Regressions

In [ ]:
columns_to_drop = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
]

X_train_model = X_train_fe.drop(columns=columns_to_drop)
X_val_model = X_val_fe.drop(columns=columns_to_drop)

# --------------------------------------------
# Define numerical and categorical features
# Avoid collinearity: FamilySize replaces SibSp/Parch; Deck replaces CabinKnown.
# --------------------------------------------

numerical_features = [
    "Age",
    "FareLog",
    "FamilySize"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
    "TicketPrefix",
    "IsAlone"
]

# --------------------------------------------
# Numerical preprocessing
# --------------------------------------------

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# --------------------------------------------
# Categorical preprocessing
# --------------------------------------------

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

# --------------------------------------------
# Combine preprocessing
# --------------------------------------------

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train_model)

# Apply the same preprocessing to validation data
X_val_processed = preprocessor.transform(X_val_model)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)

# Preprocessing and more cleaning for BernoulliNB

In [ ]:
columns_to_drop = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
]

X_train_nb = X_train_fe.drop(columns=columns_to_drop)
X_val_nb = X_val_fe.drop(columns=columns_to_drop)

numerical_features = [
    "Age",
    "FareLog",
    "FamilySize"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
    "TicketPrefix",
    "IsAlone"
]

In [ ]:
nb_numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(
        n_bins=5,
        encode="onehot",
        strategy="quantile",
        subsample=None
    ))
])

In [ ]:
nb_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
nb_preprocessor = ColumnTransformer([
    ("num", nb_numerical_pipeline, numerical_features),
    ("cat", nb_categorical_pipeline, categorical_features)
])

In [ ]:
X_train_nb_processed = nb_preprocessor.fit_transform(X_train_nb)

X_val_nb_processed = nb_preprocessor.transform(X_val_nb)

print("Training shape:", X_train_nb_processed.shape)
print("Validation shape:", X_val_nb_processed.shape)

# Train and Evaluate BernoulliNB 

In [ ]:
def evaluate_nb(alpha):
    
    # Create model
    model = BernoulliNB(alpha=alpha)
    
    # Train
    model.fit(X_train_nb_processed, y_train)
    
    # Predictions
    y_pred = model.predict(X_val_nb_processed)
    y_prob = model.predict_proba(X_val_nb_processed)[:, 1]
    
    # Metrics
    results = {
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "AUC": roc_auc_score(y_val, y_prob)
    }
    
    return model, y_pred, y_prob, results

In [ ]:
alphas = [0, 0.01, 1.0, 10]

nb_models = {}
nb_predictions = {}
nb_probabilities = {}
nb_results = []

for alpha in alphas:
    
    model, y_pred, y_prob, results = evaluate_nb(alpha)
    
    nb_models[alpha] = model
    nb_predictions[alpha] = y_pred
    nb_probabilities[alpha] = y_prob
    nb_results.append(results)

In [ ]:
nb_results_df = pd.DataFrame(nb_results)

nb_results_df

In [ ]:
fig, axes = plt.subplots(1, len(alphas), figsize=(15, 4))

for ax, alpha in zip(axes, alphas):
    
    cm = confusion_matrix(
        y_val,
        nb_predictions[alpha]
    )
    
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Did Not Survive", "Survived"],
        yticklabels=["Did Not Survive", "Survived"]
    )
    
    ax.set_title(f"BernoulliNB (α = {alpha})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for alpha in alphas:
    
    y_prob = nb_probabilities[alpha]
    
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    
    plt.plot(
        fpr,
        tpr,
        label=f"α = {alpha} (AUC = {auc:.3f})"
    )

# Random classifier
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Bernoulli Naive Bayes")
plt.legend()
plt.grid()
plt.show()

# Train and Evaluate Regressions